# HAM10000 / ISIC 2018 Baseline — Attention Residual U-Net

**Target:** Lesion Dice ≥ 0.85 (standard for ISIC 2018 Task 1 leaderboards).

## Kaggle setup
Add **all three** as inputs:
1. `kmader/skin-cancer-mnist-ham10000` (dermoscopy images, ~3 GB)
2. `tschandl/ham10000-lesion-segmentations` (binary masks)
3. `iteris-pkg` (this package)

Then: Accelerator GPU T4 x2 → Persistence Files only → Save & Run All (Commit).

**Path auto-detection** lives in Cell 1. If detection fails, edit `IMAGE_ROOTS_OVERRIDE` / `MASK_ROOT_OVERRIDE`.

## 0 · Install (no kernel restart needed)

In [ ]:
!pip install -r /kaggle/input/datasets/anfaalhossain/iteris-pkg/requirements.txt --quiet --upgrade
print('✓ Setup complete. Run All from cell 1 onwards.')

## 1 · Load config + locate the HAM10000 datasets

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '/kaggle/input/datasets/anfaalhossain/iteris-pkg')

from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config('/kaggle/input/datasets/anfaalhossain/iteris-pkg/configs/ham10000.yaml')

# Locate the two image folders + the mask folder. Auto-detect via rglob to be
# robust against Kaggle's varying mount layouts.
IMAGE_ROOTS_OVERRIDE = None    # e.g. ['/kaggle/input/.../HAM10000_images_part_1', ...]
MASK_ROOT_OVERRIDE   = None    # e.g. '/kaggle/input/.../HAM10000_segmentations_lesion_tschandl'

if IMAGE_ROOTS_OVERRIDE:
    cfg['image_roots'] = IMAGE_ROOTS_OVERRIDE
else:
    img_dirs = [p for p in Path('/kaggle/input').rglob('HAM10000_images_part_*') if p.is_dir()]
    if not img_dirs:
        raise RuntimeError('HAM10000_images_part_* folders not found. Attach kmader/skin-cancer-mnist-ham10000.')
    cfg['image_roots'] = [str(p) for p in sorted(img_dirs)]

if MASK_ROOT_OVERRIDE:
    cfg['mask_root'] = MASK_ROOT_OVERRIDE
else:
    mask_candidates = [p for p in Path('/kaggle/input').rglob('HAM10000_segmentations*') if p.is_dir()]
    if not mask_candidates:
        raise RuntimeError('HAM10000_segmentations* folder not found. Attach tschandl/ham10000-lesion-segmentations.')
    cfg['mask_root'] = str(mask_candidates[0])

cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Dataset      : {cfg["dataset"]}')
print(f'Image roots  :')
for r in cfg['image_roots']: print(f'   {r}')
print(f'Mask root    : {cfg["mask_root"]}')
print(f'Image size   : {cfg["image_size"]}  in_channels={cfg["in_channels"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Normalisation: {cfg["normalize"]}   binarise labels: {cfg["binarize_labels"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}')
print(f'Device       : {get_device()}')

## 2 · Train
Cache build is the slow part for HAM10000 — ~12 min for 10k images. Training itself is ~150 s/epoch on T4.

In [ ]:
from iteris.training import run_training

result      = run_training(cfg)
model       = result['model']
history     = result['history']
best_dice   = result['best_dice']
test_loader = result['test_loader']
checkpoint  = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.85)

## 4 · Test-set evaluation

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 5 · Export predicted masks

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 6 · Qualitative grid

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=4)

## 7 · Summary JSON

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)